## Webpage Loaders
- Load the webpage and extract the data using the `WebBaseLoader` and `BeautifulSoup` libraries.
- Use LLM to extract meaningful data from the webpage.

### Project 1: Share Market Data Analysis Based on Global Cues
- We will extract the data from the stock market website and analyze the data to understand the impact of global cues on the Indian share market.

#### Stock Market Data Extraction

In [1]:
from dotenv import load_dotenv

load_dotenv('./../.env')

True

In [2]:
from langchain_community.document_loaders import WebBaseLoader

urls = ['https://economictimes.indiatimes.com/markets/stocks/news',
        'https://www.livemint.com/latest-news',
        'https://www.livemint.com/latest-news/page-2'
        'https://www.livemint.com/latest-news/page-3',
        'https://www.moneycontrol.com/']

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader = WebBaseLoader(web_paths=urls)

In [5]:
docs = []
async for doc in loader.alazy_load():
    docs.append(doc)

Fetching pages: 100%|##########| 4/4 [00:00<00:00,  9.58it/s]


In [6]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])

In [7]:
context = format_docs(docs)

In [8]:
# print(context)
# context

import re

def text_clean(text):
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\t+', '\t', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [10]:
context = text_clean(context)

In [11]:
print(context)

Stocks in News Today - Latest News on Stocks, Stock in News | The Economic TimesBenchmarks Nifty22,512.65-601.85FEATURED FUNDS★★★★★HDFC Value Fund Direct Plan-Growth5Y Return15.82 % Invest NowFEATURED FUNDS★★★★★Motilal Oswal Midcap Fund Direct-Growth5Y Return22.19 % Invest NowEnter search text:English EditionEnglish Editionहिन्दीગુજરાતીमराठीবাংলাಕನ್ನಡമലയാളംதமிழ்తెలుగు | 23 March, 2026, 10:01 PM IST | Today's ePaper My Watchlist SubscribeSign InHomeETPrimeMarketsMarket DataMasterclassIPONewsIndustrySMEPoliticsWealthMFTechAICareersOpinionNRIPanacheMore MenuStocksNewsLive BlogStock Live BlogEarningsPodcastMarket ClassroomDons of Dalal StreetRecosStock Reports PlusNewMy ScreenerCandlestick ScreenerStock ScreenerStock WatchMarket CalendarStock Price QuotesOptionsIPOs/FPOsExpert ViewsInvestment IdeasCommoditiesViewsNewsOthersMentha OilPrecious MetalsGold MGoldSilverGold PetalSilver MicroSilver MGold GuineaSpicesCardamomOil & EnergyNatural GasCrude OilCrude Oil MiniBase MetalsAluminiumZinc Mi

#### Stock Market Data Processing with LLM

In [12]:
from scripts import llm

In [16]:
llm.qna_chain

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are helpful AI assistant who answer user question based on the provided context.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don\'t know".\n            ### Context:\n            {context}\n\n            ### Question:\n            {question}\n\n            ### Answer:'), additional_kwargs={})])
| ChatOllama(model='qwen3', base_url='http://localhost:11434')
| StrOutputParser()

In [13]:
# response = llm.ask_llm(context, "What is todays news?")
response = llm.ask_llm(context, "Extract stock market news from the given text.")


In [14]:
print(response)

Here is the extracted stock market news from the provided text:

---

### **Key Market Movements**  
- **Nifty 50 and Sensex fell sharply**, dragged by banks, auto, and consumer stocks amid **Iran-Israel tensions**, **rising crude prices**, and **rupee weakness**.  
- **Benchmarks hit 52-week lows**: 12 Sensex stocks (e.g., HDFC Bank, Kotak Mahindra Bank) fell to 52-week lows, with some slipping **21% in a month**.  
- **GIFT Nifty surged 3.6%** after **Trump paused Iran strikes**, signaling potential rebound for Indian markets.  

---

### **Corporate News**  
- **HDFC Bank crisis**: SEBI urged independent directors to act responsibly amid chairman Atanu Chakraborty’s resignation.  
- **Vedanta declared Rs 11/share interim dividend**, with a record date of **March 28**.  
- **Sula Vineyards promoter Rajeev Samant increased stake** to 23.27% via a Rs 3 crore share purchase.  
- **Adani Group stocks** (Adani Green, Adani Ports) fell due to FII stake cuts and weak earnings.  

---

### *

In [13]:
response = llm.ask_llm(context[:10_000], "Extract stock market news from the given text.")

In [14]:
print(response)

### Stock Market News Extracted from the Context:

1. **Nifty Index Performance**:  
   - Nifty closed at **25,709.85**, up **124.55** points.  
   - Nifty broke above a key chart pattern, signaling a **bullish trend**, but market breadth remains weak, and volatility has increased.  

2. **Featured Funds**:  
   - **HSBC Large Cap Fund Direct-Growth** (5Y Return: 18.34%)  
   - **UTI Aggressive Hybrid Fund Regular Plan-Growth** (5Y Return: 19.89%)  

3. **Diwali 2025 Stock Picks**:  
   - Top **10 small-cap stocks** with up to **36% upside** are highlighted for Samvat 2082.  
   - **Dixon, Nykaa, and 8 other scrips** are recommended for up to **31% gains**.  

4. **Sectoral Performance**:  
   - **Pharma, Metal, and Auto sectors** are leading the market.  
   - **Mid-caps** may see mixed performance; other sectors show improvement.  

5. **Company Actions**:  
   - **Signature Global** raised **Rs 875 crore** via non-convertible debentures to reduce debt and expand operations.  

6. **

In [17]:
# To handle large contexts, we can chunk the text and process each chunk separately.
def chunk_text(text, chunk_size, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
        
    return chunks

In [18]:

chunks = chunk_text(context, 10_000)

In [19]:


question = "Extract stock market news from the given text."

chunk_summary = []
for chunk in chunks:
    response = llm.ask_llm(chunk, question)
    chunk_summary.append(response)

In [20]:
for chunk in chunk_summary:
    print(chunk)
    print("\n\n")
    break

### Stock Market News Extracted from the Text:

1. **Market Decline**:  
   - **Nifty 50 and Sensex fell sharply** by 1,836 points, dragged by banks, auto, and consumer stocks due to **Iran-Israel tensions**, **rising crude prices**, and **rupee weakness**. Bearish momentum and elevated volatility persisted.  

2. **HDFC Bank Crisis**:  
   - **Sebi urged independent directors** to act responsibly and back insinuations with evidence after HDFC Bank chairman Atanu Chakraborty resigned.  
   - **12 Sensex stocks hit 52-week lows**, with some slipping **21% in a month**.  

3. **Market Trading Guide**:  
   - Suggested buying **Gujarat Fluorochemicals** and **HCL** for potential gains of up to **11%**.  

4. **Bulk Deals & Stock Performance**:  
   - **Ashish Kacholia exited Brand Concepts** via a ₹3.9 crore bulk deal as the stock fell **36% year-on-year**.  

5. **Large-Cap Recommendations**:  
   - Certain large-caps were labeled as **"strong buy"** or **"buy"** with **>25% upside poten

In [21]:
summary = "\n\n".join(chunk_summary)

In [22]:
# print(summary)

In [23]:
# question = "Write a detailed report in Markdown from the given context."
question = """Write a detailed market news report in markdown format. Think carefully then write the report."""
response = llm.ask_llm(summary, question)

In [24]:
import os

os.makedirs("data", exist_ok=True)

with open("data/report.md", "w", encoding="utf-8") as f:
    f.write(response)

In [25]:
with open("data/summary.md", "w", encoding="utf-8") as f:
    f.write(summary)